# 📊 Récupération des données PAC2 depuis InfluxDB

Ce notebook permet de récupérer les données d'un bucket d'InfluxDB entre deux dates spécifiées.

## Prérequis
```bash
pip install influxdb-client pandas matplotlib seaborn
```


In [1]:
# Cellule 1: Imports et dépendances
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from influxdb_client import InfluxDBClient
from influxdb_client.client.query_api import QueryApi
import warnings
warnings.filterwarnings('ignore')

# Configuration pour l'affichage dans Jupyter
plt.style.use('default')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

print("✅ Imports terminés avec succès!")

✅ Imports terminés avec succès!


In [2]:
# Cellule 2: Configuration des paramètres
# Modifiez ces valeurs selon vos besoins

# Configuration globale
output_dir = '../data'     # Répertoire de sortie (dossier parent)
bucket_name = 'AllData'    # Bucket Pervenches (PAC1 - PAC2 - AllData)
query_mode = 'daily'      # 'daily' (par jour) ou 'hourly' (par heure) si probleme de requete
data_step = '1m'          # '10s' (par seconde) ou '1h' (par heure) ou '1d' (par jour)

# Configuration des champs à récupérer
# Option 1: Champs spécifiques (recommandé pour la performance)
use_specific_fields = True  # True = champs spécifiques, False = tous les champs du bucket

# Option 2: Liste des champs spécifiques (utilisé si use_specific_fields = True)
custom_fields = [
    "sharky_energy",
    "TT_ext", "RH_ext", "price",
    "DIS2_CHD_A_TT1", "DIS2_CHD_R_TT1", "Pchd_flow",
    "DIS_VRD_retour_position_opt", "DIS_CH_A_TT1", "DIS_CH_R_TT1",
    "DIS_BEL_energy_heating",
    "PAC1_energy", "PAC2_energy", "SKID_energy",
    "PAC1_power", "PAC2_power", 
    "PAC1_cond_inlet", "PAC1_cond_outlet", "PAC1_modstatus",
    "PAC2_cond_inlet", "PAC2_cond_outlet", "PAC2_modstatus",
    "PAC1_user_pump", "PAC2_user_pump", "PAC1_domestic_pump", "PAC2_domestic_pump",
    "PAC1_general_alarm2", "PAC2_general_alarm2"
]

# Configuration des intervalles de dates
# Vous pouvez définir plusieurs intervalles à traiter automatiquement
date_intervals = [
    {
        'start_date': '2025-11-20',
        'end_date': '2025-11-27',
        'description': 'Période 1'
    },
]

print("🔧 Configuration des paramètres:")
print(f"   📁 Répertoire de sortie: {output_dir}")
print(f"   🗂️  Bucket: {bucket_name}")
print(f"   ⏰ Mode de requête: {query_mode.upper()}")
print(f"   ⏱️  Pas de temps: {data_step}")
print()
if use_specific_fields:
    print(f"   🔧 Mode champs: SPÉCIFIQUES ({len(custom_fields)} champs définis)")
    print(f"   • {', '.join(custom_fields)}")
else:
    print(f"   🔧 Mode champs: TOUS LES CHAMPS du bucket {bucket_name}")
print()
print(f"📋 {len(date_intervals)} intervalle(s) configuré(s):")
for i, interval in enumerate(date_intervals, 1):
    print(f"   {i}. {interval['description']}: {interval['start_date']} → {interval['end_date']}")
print()
print("💡 Conseils pour le mode de requête:")
print("   • 'daily': Plus rapide, recommandé pour la plupart des cas")
print("   • 'hourly': Plus lent mais plus fiable si vous avez des problèmes de serveur")
print()
print("⚠️  IMPORTANT - Configuration des champs:")
print("   • use_specific_fields = True: Utilise les champs de custom_fields")
print("   • use_specific_fields = False: Récupère TOUS les champs du bucket")
print("   • Mode 'tous les champs': Plus lent mais plus complet")
print("   • Mode 'champs spécifiques': Plus rapide et recommandé")
print("   • Exemple: ['PAC2_power', 'PAC2_energy']")
print()
print("💡 Pour ajouter d'autres intervalles:")
print("   • Ajoutez des dictionnaires dans la liste 'date_intervals'")
print("   • Chaque intervalle doit avoir 'start_date', 'end_date' et 'description'")


🔧 Configuration des paramètres:
   📁 Répertoire de sortie: ../data
   🗂️  Bucket: AllData
   ⏰ Mode de requête: DAILY
   ⏱️  Pas de temps: 1m

   🔧 Mode champs: SPÉCIFIQUES (28 champs définis)
   • sharky_energy, TT_ext, RH_ext, price, DIS2_CHD_A_TT1, DIS2_CHD_R_TT1, Pchd_flow, DIS_VRD_retour_position_opt, DIS_CH_A_TT1, DIS_CH_R_TT1, DIS_BEL_energy_heating, PAC1_energy, PAC2_energy, SKID_energy, PAC1_power, PAC2_power, PAC1_cond_inlet, PAC1_cond_outlet, PAC1_modstatus, PAC2_cond_inlet, PAC2_cond_outlet, PAC2_modstatus, PAC1_user_pump, PAC2_user_pump, PAC1_domestic_pump, PAC2_domestic_pump, PAC1_general_alarm2, PAC2_general_alarm2

📋 1 intervalle(s) configuré(s):
   1. Période 1: 2025-11-20 → 2025-11-27

💡 Conseils pour le mode de requête:
   • 'daily': Plus rapide, recommandé pour la plupart des cas
   • 'hourly': Plus lent mais plus fiable si vous avez des problèmes de serveur

⚠️  IMPORTANT - Configuration des champs:
   • use_specific_fields = True: Utilise les champs de custom_fiel

In [3]:
# Cellule 3: Fonction principale de récupération des données

def build_flux_query(bucket, start_time, stop_time, use_specific_fields, fields=None, bucket_name=None, data_step='1h'):
    """
    Génère une requête Flux selon les paramètres.
    
    Args:
        bucket (str): Nom du bucket InfluxDB
        start_time (str): Heure de début au format ISO
        stop_time (str): Heure de fin au format ISO
        use_specific_fields (bool): True pour champs spécifiques, False pour tous les champs
        fields (list): Liste des champs (si use_specific_fields=True)
        bucket_name (str): Nom du bucket pour le filtre measurement (si use_specific_fields=False)
        data_step (str): Pas de temps pour l'agrégation
    
    Returns:
        str: Requête Flux complète
    """
    if use_specific_fields:
        fields_filter = ' or '.join([f'r["_field"] == "{field}"' for field in fields])
        filter_condition = fields_filter
    else:
        filter_condition = f'r._measurement == "{bucket_name}"'
    
    return (
        f'from(bucket: "{bucket}") '
        f'|> range(start: {start_time}, stop: {stop_time})'
        f'|> filter(fn: (r) => {filter_condition})'
        f'|> drop(columns: ["_measurement", "result", "table", "_start", "_stop"])'
        f'|> aggregateWindow(every: {data_step}, fn: last, createEmpty: false)'
        f'|> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")'
        f'|> yield(name: "pivoted")'
    )

def process_period(query_api, bucket, start_time, stop_time, use_specific_fields, fields, bucket_name, data_step, period_name):
    """
    Traite une période (heure ou jour) et retourne les données.
    
    Args:
        query_api: API de requête InfluxDB
        bucket (str): Nom du bucket
        start_time (str): Heure de début
        stop_time (str): Heure de fin
        use_specific_fields (bool): Mode de champs
        fields (list): Champs spécifiques
        bucket_name (str): Nom du bucket pour le filtre
        data_step (str): Pas de temps
        period_name (str): Nom de la période pour l'affichage
    
    Returns:
        pandas.DataFrame ou None: Données récupérées ou None si vide
    """
    flux_query = build_flux_query(bucket, start_time, stop_time, use_specific_fields, fields, bucket_name, data_step)
    df_period = query_api.query_data_frame(flux_query)
    df_period = df_period.loc[:, ~df_period.columns.str.startswith('result')]
    
    if not df_period.empty:
        print(f"   ✅ {len(df_period):,} lignes récupérées")
        return df_period
    else:
        print(f"   ⚠️  Aucune donnée pour {period_name}")
        return None

def get_data(start_date, end_date, fields=None, use_specific_fields=True, output_dir='./data', query_mode='daily', data_step='1h'):
    """
    Récupère les données InfluxDB entre deux dates spécifiées.
    La requête peut être divisée par jour ou par heure pour éviter les problèmes de serveur.
    
    Args:
        start_date (str): Date de début au format 'YYYY-MM-DD'
        end_date (str): Date de fin au format 'YYYY-MM-DD'
        fields (list): Liste des champs à récupérer (obligatoire si use_specific_fields=True)
        use_specific_fields (bool): True pour champs spécifiques, False pour tous les champs
        output_dir (str): Répertoire de sortie (défaut: './data')
        query_mode (str): Mode de requête 'daily' ou 'hourly' (défaut: 'daily')
        data_step (str): Pas de temps '10s', '1h', '1d' (défaut: '1h')
    
    Returns:
        pandas.DataFrame: DataFrame avec les données récupérées
        
    Raises:
        ValueError: Si les champs sont manquants, vides ou invalides
    """
    print(f"🚀 Début de la récupération des données {bucket_name}...")
    print("=" * 50)
    
    # Validation et conversion des dates
    try:
        start_dt = pd.to_datetime(start_date)
        end_dt = pd.to_datetime(end_date)
    except Exception as e:
        raise ValueError(f"❌ Format de date invalide. Utilisez le format 'YYYY-MM-DD'. Erreur: {e}")
    
    if start_dt >= end_dt:
        raise ValueError("❌ La date de début doit être antérieure à la date de fin.")
    
    # Configuration pour le bucket
    output_filename = f'data_{bucket_name}_{start_date}_to_{end_date}.csv'
    
    # Validation selon le mode choisi
    if use_specific_fields:
        # Mode champs spécifiques - validation obligatoire
        if fields is None or len(fields) == 0:
            raise ValueError("❌ Erreur: Les champs (fields) sont obligatoires quand use_specific_fields=True.")
        
        # Validation que les champs sont des chaînes de caractères
        if not all(isinstance(field, str) for field in fields):
            raise ValueError("❌ Erreur: Tous les champs doivent être des chaînes de caractères (strings).")
        
        # Validation que les champs ne sont pas vides
        if not all(field.strip() for field in fields):
            raise ValueError("❌ Erreur: Les champs ne peuvent pas être vides ou contenir uniquement des espaces.")
        
        # Validation du format des champs (pas de caractères spéciaux dangereux)
        import re
        invalid_chars = re.compile(r'[<>"\'\s]')
        for field in fields:
            if invalid_chars.search(field):
                raise ValueError(f"❌ Erreur: Le champ '{field}' contient des caractères invalides. Utilisez uniquement des lettres, chiffres et underscores.")
        
        print(f"✅ Mode champs spécifiques: {len(fields)} champs validés")
    else:
        # Mode tous les champs - pas de validation nécessaire
        print(f"✅ Mode tous les champs: récupération de tous les champs du bucket {bucket_name}")
    
    print(f"📊 Récupération des données {bucket_name}")
    print(f"📅 Période: {start_dt.strftime('%d/%m/%Y')} → {end_dt.strftime('%d/%m/%Y')}")
    print(f"📁 Fichier de sortie: {output_filename}")
    if use_specific_fields:
        print(f"🔧 Champs récupérés: {len(fields)} champs spécifiques")
        print(f"   • {', '.join(fields)}")
    else:
        print(f"🔧 Champs récupérés: TOUS les champs du bucket {bucket_name}")
    print(f"⏰ Mode de requête: {query_mode.upper()}")
    print()

    # InfluxDB credentials
    url = 'https://eu-central-1-1.aws.cloud2.influxdata.com'
    token = 'asV9ER3o1xmKTQMjTT_km59aDFlhgvwXolZXeEvnvZde_F2BQEjbdqAfKiMN-71UNuSHLqlTJLNoT2ueNr-gzg=='
    org = 'Pervenches'
    bucket = bucket_name

    # Connexion au client InfluxDB
    client = InfluxDBClient(url=url, token=token, org=org)
    query_api = client.query_api()

    # Génération de la liste des périodes à traiter selon le mode
    if query_mode.lower() == 'hourly':
        date_range = pd.date_range(start=start_dt, end=end_dt, freq='H')
        period_name = "heure"
        print(f"⏰ Traitement de {len(date_range)} heure(s) séparément pour éviter les problèmes de serveur")
    else:
        date_range = pd.date_range(start=start_dt, end=end_dt, freq='D')
        period_name = "jour"
        print(f"📅 Traitement de {len(date_range)} jour(s) séparément pour éviter les problèmes de serveur")
    
    print()
    
    # Liste pour accumuler les DataFrames de chaque période
    df_list = []
    
    # Traitement unifié pour toutes les périodes
    for i, current_time in enumerate(date_range, 1):
        if query_mode.lower() == 'hourly':
            start_time = current_time.strftime('%Y-%m-%dT%H:00:00Z')
            stop_time = (current_time + pd.Timedelta(hours=1)).strftime('%Y-%m-%dT%H:00:00Z')
            display_name = current_time.strftime('%d/%m/%Y %H:00')
        else:
            start_time = current_time.strftime('%Y-%m-%dT00:00:00Z')
            stop_time = (current_time + pd.Timedelta(days=1)).strftime('%Y-%m-%dT00:00:00Z')
            display_name = current_time.strftime('%d/%m/%Y')
        
        print(f"🔄 {period_name.title()} {i}/{len(date_range)}: {display_name}")
        
        # Traitement de la période avec la fonction unifiée
        df_period = process_period(
            query_api, bucket, start_time, stop_time, 
            use_specific_fields, fields, bucket_name, data_step, period_name
        )
        
        if df_period is not None:
            df_list.append(df_period)
        print()

    # Fusion de tous les DataFrames
    if df_list:
        df_final = pd.concat(df_list, ignore_index=True)
        print(f"📊 Total: {len(df_final):,} lignes récupérées sur {len(date_range)} {period_name}(s)")
    else:
        df_final = pd.DataFrame()
        print("❌ Aucune donnée récupérée pour la période spécifiée.")

    # Sauvegarde des données
    if not df_final.empty:
        os.makedirs(output_dir, exist_ok=True)
        filename = os.path.join(output_dir, output_filename)
        df_final.to_csv(filename, index=False)
        
        print("✅ Données récupérées avec succès!")
        print(f"📊 Nombre de lignes: {len(df_final):,}")
        print(f"📁 Fichier sauvegardé: {filename}")
        print(f"📅 Période couverte: {df_final['_time'].min()} → {df_final['_time'].max()}")
        print(f" Pas de temps: {data_step}")
        print()
        
        # Affichage des premières lignes
        print("📋 Aperçu des données:")
        display(df_final.head())
        print()
        
        # Informations sur les colonnes
        print("📈 Colonnes disponibles:")
        for col in df_final.columns:
            if col != '_time':
                non_null_count = df_final[col].notna().sum()
                print(f"   • {col}: {non_null_count:,} valeurs non-nulles")
        print()
    
    return df_final

def process_multiple_intervals(date_intervals, bucket_name, output_dir, query_mode, data_step, custom_fields, use_specific_fields):
    """
    Traite plusieurs intervalles de dates automatiquement.
    """
    print("🚀 Début du traitement de plusieurs intervalles")
    print("=" * 60)
    
    results = []
    total_intervals = len(date_intervals)
    
    for i, interval in enumerate(date_intervals, 1):
        print(f"\n📅 Traitement de l'intervalle {i}/{total_intervals}")
        print(f"   📝 Description: {interval['description']}")
        print(f"   📅 Période: {interval['start_date']} → {interval['end_date']}")
        print("-" * 50)
        
        try:
            # Appel de la fonction get_data pour cet intervalle
            df_result = get_data(
                start_date=interval['start_date'],
                end_date=interval['end_date'],
                fields=custom_fields,
                use_specific_fields=use_specific_fields,
                output_dir=output_dir,
                query_mode=query_mode,
                data_step=data_step
            )
            
            # Stockage du résultat
            result = {
                'interval': interval,
                'dataframe': df_result,
                'success': True,
                'error': None
            }
            results.append(result)
            
            print(f"✅ Intervalle {i} traité avec succès!")
            
        except Exception as e:
            print(f"❌ Erreur lors du traitement de l'intervalle {i}: {str(e)}")
            
            # Stockage de l'erreur
            result = {
                'interval': interval,
                'dataframe': None,
                'success': False,
                'error': str(e)
            }
            results.append(result)
    
    # Résumé final
    print("\n" + "=" * 60)
    print("📊 RÉSUMÉ DU TRAITEMENT")
    print("=" * 60)
    
    successful = sum(1 for r in results if r['success'])
    failed = total_intervals - successful
    
    print(f"✅ Intervalles traités avec succès: {successful}/{total_intervals}")
    print(f"❌ Intervalles en erreur: {failed}/{total_intervals}")
    
    if failed > 0:
        print("\n❌ Intervalles en erreur:")
        for i, result in enumerate(results):
            if not result['success']:
                print(f"   • {result['interval']['description']}: {result['error']}")
    
    print(f"\n📁 Fichiers sauvegardés dans: {output_dir}")
    print("🎉 Traitement terminé!")
    
    return results



In [ ]:
# Cellule 4: Exécution de la récupération des données et concaténation
# Exécutez cette cellule pour traiter tous les intervalles configurés

import json
from datetime import datetime

print("🚀 Lancement du traitement de plusieurs intervalles...")
print()

# Traitement de tous les intervalles
results = process_multiple_intervals(date_intervals, bucket_name, output_dir, query_mode, data_step, custom_fields, use_specific_fields)

# Affichage du résumé détaillé
print("\n📋 RÉSUMÉ DÉTAILLÉ DES RÉSULTATS")
print("=" * 50)

successful_dataframes = []
total_rows = 0

for i, result in enumerate(results, 1):
    interval = result['interval']
    print(f"\n{i}. {interval['description']}")
    print(f"   📅 Période: {interval['start_date']} → {interval['end_date']}")
    
    if result['success']:
        df = result['dataframe']
        if df is not None and not df.empty:
            print(f"   ✅ Succès: {len(df):,} lignes récupérées")
            print(f"   📅 Couverture: {df['_time'].min()} → {df['_time'].max()}")
            successful_dataframes.append(df)
            total_rows += len(df)
        else:
            print(f"   ⚠️  Succès mais aucune donnée récupérées")
    else:
        print(f"   ❌ Erreur: {result['error']}")

# Concaténation des DataFrames réussis
if successful_dataframes:
    print(f"\n🔄 Concaténation des {len(successful_dataframes)} intervalle(s) réussis...")
    
    # Concaténation de tous les DataFrames
    df_combined = pd.concat(successful_dataframes, ignore_index=True)
    
    # Tri par timestamp pour avoir les données dans l'ordre chronologique
    df_combined = df_combined.sort_values('_time').reset_index(drop=True)
    
    # Génération du nom de fichier pour le fichier combiné
    all_start_dates = [interval['start_date'] for interval in date_intervals]
    all_end_dates = [interval['end_date'] for interval in date_intervals]
    combined_filename = f'data_{bucket_name}_combined_{min(all_start_dates)}_to_{max(all_end_dates)}.csv'
    combined_filepath = os.path.join(output_dir, combined_filename)
    
    # Sauvegarde du fichier combiné
    df_combined.to_csv(combined_filepath, index=False)
    
    print(f"✅ Fichier combiné créé: {combined_filename}")
    print(f"📊 Total combiné: {len(df_combined):,} lignes")
    print(f"📅 Période totale: {df_combined['_time'].min()} → {df_combined['_time'].max()}")
    
    # Création du fichier de métadonnées JSON
    metadata = {
        "generation_info": {
            "created_at": datetime.now().isoformat(),
            "notebook_version": "1.0",
            "description": "Métadonnées de la récupération de données InfluxDB"
        },
        "configuration": {
            "bucket_name": bucket_name,
            "output_dir": output_dir,
            "query_mode": query_mode,
            "data_step": data_step,
            "use_specific_fields": use_specific_fields,
            "custom_fields": custom_fields if use_specific_fields else "ALL_FIELDS"
        },
        "intervals": [
            {
                "description": interval['description'],
                "start_date": interval['start_date'],
                "end_date": interval['end_date'],
                "success": result['success'],
                "error": result['error'] if not result['success'] else None,
                "rows_count": len(result['dataframe']) if result['success'] and result['dataframe'] is not None else 0,
                "time_range": {
                    "start": str(result['dataframe']['_time'].min()) if result['success'] and result['dataframe'] is not None and not result['dataframe'].empty else None,
                    "end": str(result['dataframe']['_time'].max()) if result['success'] and result['dataframe'] is not None and not result['dataframe'].empty else None
                }
            }
            for interval, result in zip(date_intervals, results)
        ],
        "combined_file": {
            "filename": combined_filename,
            "filepath": combined_filepath,
            "total_rows": len(df_combined),
            "total_intervals": len(successful_dataframes),
            "time_range": {
                "start": str(df_combined['_time'].min()),
                "end": str(df_combined['_time'].max())
            },
            "columns": list(df_combined.columns),
            "columns_count": len(df_combined.columns)
        },
        "statistics": {
            "total_intervals_requested": len(date_intervals),
            "successful_intervals": len(successful_dataframes),
            "failed_intervals": len(date_intervals) - len(successful_dataframes),
            "success_rate": f"{(len(successful_dataframes) / len(date_intervals) * 100):.1f}%"
        }
    }
    
    # Sauvegarde du fichier de métadonnées
    metadata_filename = f'metadata_{bucket_name}_{min(all_start_dates)}_to_{max(all_end_dates)}.json'
    metadata_filepath = os.path.join(output_dir, metadata_filename)
    
    with open(metadata_filepath, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    
    print(f"📄 Fichier de métadonnées créé: {metadata_filename}")
    print(f"📁 Répertoire: {output_dir}")
    
    # Affichage des statistiques finales
    print(f"\n📊 STATISTIQUES FINALES")
    print("=" * 40)
    print(f"✅ Intervalles réussis: {len(successful_dataframes)}/{len(date_intervals)}")
    print(f"📈 Taux de succès: {(len(successful_dataframes) / len(date_intervals) * 100):.1f}%")
    print(f"📊 Total de lignes: {len(df_combined):,}")
    print(f"📅 Période couverte: {df_combined['_time'].min()} → {df_combined['_time'].max()}")
    print(f"🔧 Colonnes: {len(df_combined.columns)}")
    
    # Affichage des colonnes disponibles
    print(f"\n📋 Colonnes disponibles:")
    for col in df_combined.columns:
        if col != '_time':
            non_null_count = df_combined[col].notna().sum()
            print(f"   • {col}: {non_null_count:,} valeurs non-nulles")
    
    print(f"\n💡 Fichiers générés:")
    print(f"   📄 {combined_filename} - Données combinées")
    print(f"   📄 {metadata_filename} - Métadonnées et configuration")
    
else:
    print("\n❌ Aucune donnée à concaténer - tous les intervalles ont échoué")

print("\n💡 Conseils pour l'utilisation:")
print("   • Le fichier combiné contient toutes les données dans l'ordre chronologique")
print("   • Le fichier JSON contient tous les paramètres et statistiques")
print("   • Modifiez 'date_intervals' dans la cellule 2 pour personnaliser")


🚀 Lancement du traitement de plusieurs intervalles...

🚀 Début du traitement de plusieurs intervalles

📅 Traitement de l'intervalle 1/1
   📝 Description: Période 1
   📅 Période: 2025-11-20 → 2025-11-27
--------------------------------------------------
🚀 Début de la récupération des données AllData...
✅ Mode champs spécifiques: 28 champs validés
📊 Récupération des données AllData
📅 Période: 20/11/2025 → 27/11/2025
📁 Fichier de sortie: data_AllData_2025-11-20_to_2025-11-27.csv
🔧 Champs récupérés: 28 champs spécifiques
   • sharky_energy, TT_ext, RH_ext, price, DIS2_CHD_A_TT1, DIS2_CHD_R_TT1, Pchd_flow, DIS_VRD_retour_position_opt, DIS_CH_A_TT1, DIS_CH_R_TT1, DIS_BEL_energy_heating, PAC1_energy, PAC2_energy, SKID_energy, PAC1_power, PAC2_power, PAC1_cond_inlet, PAC1_cond_outlet, PAC1_modstatus, PAC2_cond_inlet, PAC2_cond_outlet, PAC2_modstatus, PAC1_user_pump, PAC2_user_pump, PAC1_domestic_pump, PAC2_domestic_pump, PAC1_general_alarm2, PAC2_general_alarm2
⏰ Mode de requête: DAILY

📅 Trai

In [ ]:
# Cellule 5: Exemple de traitement d'un intervalle individuel
# Utilisez cette cellule si vous voulez traiter un seul intervalle spécifique

# Configuration pour un intervalle individuel
start_date_individual = '2025-07-25'      # Date de début
end_date_individual = '2025-07-26'        # Date de fin

print("📅 Configuration pour un intervalle individuel:")
print(f"   📅 Période: {start_date_individual} → {end_date_individual}")
print(f"   🗂️  Bucket: {bucket_name}")
print(f"   ⏰ Mode: {query_mode.upper()}")
print(f"   ⏱️  Pas de temps: {data_step}")
print(f"   🔧 Champs: {len(custom_fields)} champs")
print()

# Décommentez la ligne suivante pour exécuter le traitement individuel
# df_individual = get_data(start_date_individual, end_date_individual, custom_fields, use_specific_fields, output_dir, query_mode, data_step)

# Exemple avec des champs personnalisés
# custom_fields_individual = ["PAC2_power", "PAC2_energy"]
# df_individual = get_data(start_date_individual, end_date_individual, custom_fields_individual, True, output_dir, query_mode, data_step)

# Exemple pour récupérer tous les champs du bucket
# df_all_fields = get_data(start_date_individual, end_date_individual, None, False, output_dir, query_mode, data_step)

print("💡 Pour traiter cet intervalle individuel:")
print("   1. Décommentez la ligne d'exécution ci-dessus")
print("   2. Modifiez les paramètres selon vos besoins")
print("   3. Exécutez la cellule")


In [ ]:
# Cellule 6: Détection des valeurs aberrantes (optionnel)
# Utilisez cette cellule pour analyser les données récupérées

import numpy as np

def detect_outliers(df, method='iqr', threshold=1.5, show_details=True):
    """
    Détecte les valeurs aberrantes dans un DataFrame.
    
    Args:
        df: DataFrame pandas
        method: 'iqr', 'zscore', ou 'modified_zscore'
        threshold: Seuil pour la détection
        show_details: Afficher les détails
    
    Returns:
        dict: Dictionnaire avec les outliers détectés
    """
    outliers = {}
    
    for column in df.select_dtypes(include=[np.number]).columns:
        if column == '_time':
            continue
            
        data = df[column].dropna()
        
        if method == 'iqr':
            Q1 = data.quantile(0.25)
            Q3 = data.quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - threshold * IQR
            upper_bound = Q3 + threshold * IQR
            outliers[column] = data[(data < lower_bound) | (data > upper_bound)]
            
        elif method == 'zscore':
            z_scores = np.abs((data - data.mean()) / data.std())
            outliers[column] = data[z_scores > threshold]
            
        elif method == 'modified_zscore':
            median = data.median()
            mad = np.median(np.abs(data - median))
            modified_z_scores = 0.6745 * (data - median) / mad
            outliers[column] = data[np.abs(modified_z_scores) > threshold]
    
    if show_details:
        print(f"🔍 Détection des valeurs aberrantes (méthode: {method})")
        print("=" * 50)
        
        total_outliers = 0
        for column, outlier_data in outliers.items():
            if len(outlier_data) > 0:
                print(f"📊 {column}: {len(outlier_data)} valeurs aberrantes")
                total_outliers += len(outlier_data)
        
        if total_outliers == 0:
            print("✅ Aucune valeur aberrante détectée")
        else:
            print(f"⚠️  Total: {total_outliers} valeurs aberrantes détectées")
    
    return outliers

print("✅ Fonction de détection des valeurs aberrantes créée!")
print("💡 Utilisez 'detect_outliers(df)' pour analyser vos données")


In [ ]:
# Cellule 7: Analyse des valeurs aberrantes sur les données récupérées
# Utilisez cette cellule pour analyser les données après récupération

# Exemple d'analyse des valeurs aberrantes
# Décommentez et modifiez selon vos besoins

# # Analyse des valeurs aberrantes sur le premier intervalle récupéré
if 'results' in locals() and len(results) > 0:
    first_result = results[0]
    if first_result['success'] and first_result['dataframe'] is not None:
        df_to_analyze = first_result['dataframe']
        
        print("🔍 Analyse des valeurs aberrantes sur le premier intervalle:")
        print(f"   📅 Période: {first_result['interval']['description']}")
        print(f"   📊 Données: {len(df_to_analyze):,} lignes")
        print()
        
        # Détection avec différentes méthodes
        outliers_iqr = detect_outliers(df_to_analyze, method='iqr', threshold=1.5)
        outliers_zscore = detect_outliers(df_to_analyze, method='zscore', threshold=3)
        
        print("\n💡 Conseils pour l'analyse:")
        print("   • Méthode IQR: Détecte les valeurs en dehors de 1.5x l'écart interquartile")
        print("   • Méthode Z-score: Détecte les valeurs avec |z| > 3")
        print("   • Modifiez les seuils selon vos besoins")

print("💡 Pour analyser les valeurs aberrantes:")
print("   1. Exécutez d'abord la cellule 4 pour récupérer les données")
print("   2. Décommentez le code ci-dessus")
print("   3. Modifiez les paramètres selon vos besoins")
print("   4. Exécutez cette cellule")


In [ ]:
# Cellule 8: Exemple de traitement d'un intervalle individuel
# Utilisez cette cellule si vous voulez traiter un seul intervalle spécifique

# Configuration pour un intervalle individuel
individual_config = {
    'start_date': '2025-07-25',      # Date de début
    'end_date': '2025-07-26',        # Date de fin
    'bucket_name': 'AllData',        # Bucket à utiliser
    'output_dir': './data',          # Répertoire de sortie
    'query_mode': 'daily',           # Mode de requête ('daily' ou 'hourly')
    'data_step': '1h',               # Pas de temps ('10s', '1h', '1d')
    'custom_fields': [               # Champs à récupérer
        "PAC2_power",
        "PAC2_energy"
    ]
}

print("📅 Configuration pour un intervalle individuel:")
print(f"   📅 Période: {individual_config['start_date']} → {individual_config['end_date']}")
print(f"   🗂️  Bucket: {individual_config['bucket_name']}")
print(f"   ⏰ Mode: {individual_config['query_mode'].upper()}")
print(f"   ⏱️  Pas de temps: {individual_config['data_step']}")
print(f"   🔧 Champs: {len(individual_config['custom_fields'])} champs")
print()

# Décommentez la ligne suivante pour exécuter le traitement individuel
# df_individual = get_pac2_data(
#     start_date=individual_config['start_date'],
#     end_date=individual_config['end_date'],
#     fields=individual_config['custom_fields'],
#     output_dir=individual_config['output_dir'],
#     query_mode=individual_config['query_mode'],
#     data_step=individual_config['data_step']
# )

print("💡 Pour traiter cet intervalle individuel:")
print("   1. Décommentez la ligne d'exécution ci-dessus")
print("   2. Modifiez les paramètres selon vos besoins")
print("   3. Exécutez la cellule")


## 📚 Guide d'utilisation - Intervalles multiples

### 🎯 **Nouvelles fonctionnalités ajoutées :**

#### **1. Configuration flexible du bucket et du pas de temps :**
- **`bucket_name`** : Choisissez entre `'AllData'`, `'PAC1'`, `'PAC2'`
- **`data_step`** : Contrôlez la granularité des données (`'10s'`, `'1h'`, `'1d'`)

#### **2. Traitement de plusieurs intervalles :**
- **Configuration simple** : Définissez plusieurs périodes dans `date_intervals`
- **Traitement automatique** : Tous les intervalles sont traités séquentiellement
- **Gestion d'erreurs** : Continue même si un intervalle échoue
- **Résumé détaillé** : Affichage des résultats pour chaque intervalle

### 🔧 **Comment utiliser :**

#### **Pour traiter plusieurs intervalles :**
1. **Modifiez la cellule 5** : Ajoutez vos intervalles dans `date_intervals`
2. **Configurez les paramètres** : Ajustez `global_config` selon vos besoins
3. **Exécutez la cellule 7** : Lance le traitement automatique

#### **Pour traiter un intervalle unique :**
1. **Utilisez la cellule 4** : Pour un intervalle simple
2. **Ou la cellule 8** : Pour un intervalle avec configuration personnalisée

### 📋 **Exemple de configuration :**

```python
# Plusieurs intervalles
date_intervals = [
    {
        'start_date': '2025-07-19',
        'end_date': '2025-07-21',
        'description': 'Période 1 - Juillet 19-21'
    },
    {
        'start_date': '2025-08-01',
        'end_date': '2025-08-03',
        'description': 'Période 2 - Août 1-3'
    }
]

# Configuration globale
global_config = {
    'bucket_name': 'AllData',
    'query_mode': 'hourly',
    'data_step': '10s',
    'custom_fields': ['PAC2_power', 'PAC2_energy']
}
```

### 💡 **Conseils d'utilisation :**

- **Mode `hourly`** : Plus fiable pour les serveurs sensibles
- **Mode `daily`** : Plus rapide pour les grandes périodes
- **Pas de temps `10s`** : Données très détaillées (attention au volume)
- **Pas de temps `1h`** : Bon compromis entre détail et performance
- **Pas de temps `1d`** : Pour des analyses à long terme

### 🚨 **Gestion des erreurs :**
- Le traitement continue même si un intervalle échoue
- Un résumé détaillé indique les succès et échecs
- Chaque intervalle génère son propre fichier CSV
